# Module 5: Decision Trees

---

## What is a Decision Tree?

A **Decision Tree** is a supervised machine learning algorithm used primarily for **classification** problems, primarily dealing with categorical features.

It models decisions as a **tree structure**, where:
- Each **internal node** tests a condition on a feature
- Each **branch** represents the outcome of that test
- Each **leaf node** gives a final prediction (class label)

**Intuition:** It mimics how humans make decisions — by asking a series of yes/no questions until a conclusion is reached.

---

## Structure of a Decision Tree

### 1. Root Node
- The **starting point** of the tree
- Contains the **entire training dataset**
- The first split is made here, on the feature that best separates the data

### 2. Splitting
- The process of **dividing a node's data into subsets** based on a condition on a feature
- **Why split?** To produce subsets that are more **homogeneous** (purer) in class labels than the parent node
- **Which feature to split on?** Determined by a **splitting criterion**:
  - **Gini Impurity** — measures the probability of misclassifying a randomly chosen sample
  - **Entropy / Information Gain** — measures the reduction in uncertainty after a split
- **How many branches?**
  - Categorical feature → one branch per distinct value
  - Numerical feature → typically two branches using a threshold (e.g., age ≤ 30 vs. age > 30)

> A **perfectly homogeneous** (pure) node contains samples from **only one class**.

### 3. Branch Node (Decision Node / Internal Node)
- A node that **is not a leaf** — it still has children
- Data is routed down a branch based on a **condition on a specific feature**
- Example: *"Is income > $50K?"* → routes samples left or right

### 4. Leaf Node (Terminal Node)
- The **endpoints** of the tree — no further splitting occurs
- Each leaf assigns a **predicted class label**, typically the **majority class** of the samples that reached it
- **Pure leaf:** All samples belong to one class → ideal outcome
- **Impure leaf:** Samples from multiple classes → prediction is still made, but with less confidence

### 5. Sub-Tree
- Any **node together with all nodes below it** forms a sub-tree
- The root of a sub-tree is the branch node from which it originates

### 6. Pruning
- The process of **removing branches or sub-trees** that provide little predictive power
- **Why prune?** An unpruned tree will keep splitting until every leaf is pure
— this causes **overfitting**: the model memorizes training data and fails to generalize to new data
- **Types:**
  - **Pre-pruning (Early Stopping):** Stop growing the tree early based on a condition (e.g., max depth, min samples per leaf)
  - **Post-pruning:** Grow the full tree first, then remove branches that do not improve performance on validation data

### How Decision Tree Make Prediction

![Image Description](DIKHSITATREE.png)

Gievn: **Segment = 0**, **Num_of_Promotions = 1**  
Trying to Predict: **Units_sold>1000 = ??**

```python
Root: Segment <= 1.5?   → 0 <= 1.5 → TRUE  → go left
Node: Segment <= 0.5?   → 0 <= 0.5 → TRUE  → go left
Node: Num_of_Promotions <= 0.5?  → 1 <= 0.5 → FALSE → go right
Leaf: gini=0.0, samples=1, value=[1,0], class=Low Sales
```
Predicted: **Low Sales (0)**


##### But...
- The core problem: the tree memorized instead of learned

| Situation | What it means |
|-|-|
| Large Leaf (ex. samples = 15) | Prediction based on many examples → more trustworthy |
| Small Leaf (ex. samples = 4) | Somewhat risky |
| Leaf with sample = 1 | The tree **memorized one data point — high overfitting risk** |

**Solution**: Pruning

1. **Pre-pruning** — set constraints before training:
    - *min_samples_leaf = 5* → no leaf is allowed to have fewer than 5 samples
    - *max_depth = 3* → tree can't go deeper than 3 levels

2. **Post-pruning** — grow the full tree, then collapse leaves that don't improve validation performance



### Hyperparameter



|| Parameters | Hyperparameters |
|-|-|-|
| What it is | Values the model *learns* from data | Values you *set* before training begins |
|When it's set | The algorithm, automatically | **You, the human**
| Example (Decision Tree) | Which feature to split on, which threshold to use | Max depth, min samples per leaf

Think of training a decision tree like growing a plant:

- The plant decides on its own how to grow its branches (those are **parameters** — learned from data)
- But you decide: how tall to let it grow, when to prune it, how much water to give it — those are **hyperparameters** — constraints you impose before and during growth.

The plant's actual shape is learned. The rules you impose on it are hyperparameters.

#### Key Hyperparameters

1. **criterion**

    - Impurity measure to use when deciding the best split
    - Options: 
        - **"Gini Impurity or Gini Index"** (Default)
        - **Entropy**
        - **Logarithmic Loss (Log Loss)** (For classification problem)

2. **splitter**

    - **splitter="best" (default)**
        - At each node, the tree considers every single feature and every possible threshold
        - It then picks the globally best split among all of them
        - Result: the most optimal tree for the training data
        - Cost: slower, especially with many features

    - **splitter="random"**
        - At each node, the tree picks a random subset of features and finds the best split within that random subset only
        - It does not consider all features — just a random sample of them
        - Result: a less optimal but faster tree; also introduces randomness into the structure


3. **random_state**

    - Allow the random number generator's randomness to be consistent without removing randomness from the algorithm

4. **max_depth**
    - Limits how many levels deep the tree can grow
    - Example: *max_depth=3* means the tree can ask at most 3 questions before making a prediction.

5. **min_samples_split**
    - Minimum samples a node must have *before it's even allowed to split*
    - If a node has fewer samples than this, it becomes a leaf **immediately**

6. **min_samples_leaf**
    - The minimum number of samples a leaf node must have
    - Example: *min_samples_leaf=5* → no leaf can be  created from fewer than 5 training samples.

### Handle Imbalanced Datasets



#### Why Imbalanced Datasets Are a Problem First
Suppose your dataset has **950 "No"** and **50 "Yes"** samples. A model that predicts "No" for everything achieves **95% accuracy** — yet it has learned nothing useful. This is why imbalanced data is dangerous, and why it needs special handling.

1. Using Appropriate **Evaluation Metrics**
    - Accuracy alone is misleading on imbalanced data. Better metrics:
        - Precision
        - Recall
        - F1-Score
        - ROC-AUC
    - **Rule**: When classes are imbalanced, always report F1 or AUC — never just accuracy.

2. **Stratified Split**

    - Ensures the **proportion of each class** in training and test sets matches the proportion in the original dataset

    - It does **not** make classes equal in size — it preserves the original ratio

```python
# Example
Original data: 90% No, 10% Yes (100 samples)

Without stratification (bad luck):
  Train: 95% No, 5% Yes  ← minority class underrepresented
  Test:  85% No, 15% Yes ← skewed evaluation

With stratification (always correct):
  Train: 90% No, 10% Yes ← same ratio as original
  Test:  90% No, 10% Yes ← same ratio as original
  ```
3. ***class_weight* Hyperparameter**

class_weight* tells the model: "**treat mistakes on the minority class as more costly**"

*   **`class_weight='balanced'`**
    *   The minority class automatically receives a **higher weight**.
    *   Errors in the minority class penalize the model more heavily during training.
*   **`class_weight={0: 0.8, 1: 0.2}` (Manual Weighting)**
    *   Allows you to manually assign the importance of each class.
    *   *Example:* Here, Class 0 mistakes are penalized **4× more** than Class 1.
    *   Best used when you have domain knowledge regarding the specific "cost" of an error.
> Key insight: class_weight does not change your data — it changes how the model prioritizes learning during training.

4. **Undersampling vs. Oversampling**

*   **`Undersampling — Shrink the Majority`**
    - You discard majority class samples randomly
    - **Problem**: those discarded samples contained real information about the world

```python
# Example
Before:  [■■■■■■■■■■ Majority (1000)] [● Minority (100)]
After:   [■■■■■■■■■■ Majority (100)]  [● Minority (100)]
```

*   **`Oversampling — Grow the Minority`**
    - **Random Oversampling**: Copy existing minority samples repeatedly
        - No data loss, but the model sees the exact same minority examples many times → memorizes them → **overfitting**
    - **Synthetic Minority Oversampling Technique (SMOTE)**: Creates samples that are **plausible interpolations** of real data — not just copies.

**Random Oversampling**: Copy existing minority samples repeatedly

```python
# Example
Before:  [■■■■■■■■■■ Majority (1000)] [● Minority (100)]
After:   [■■■■■■■■■■ Majority (1000)] [●●●●●●●●●● Minority (1000)]
```

### Full Comparison of Four Techniques

| Technique | What it changes | Risk |
|-|-|-|
| Stratified split | How data is divided | None — just better practice |
| Class weights | How splits are scored internally | None — data unchanged |
| Undersampling | Removes majority rows | Information loss |
| Random Oversampling | Duplicates minority rows | Overfitting on repeated samples |
| SMOTE | Generates synthetic minority rows	 | Synthetic data may not be realistic |

**Practical rule of thumb**:
- Large dataset → try class weights or SMOTE first
- Small dataset → avoid undersampling (can't afford data loss)
- Always compare test F1 scores across all techniques to pick the winner